In [60]:
import pandas as pd
import numpy as np

np.random.seed(42)

In [71]:
# 1: Create and Define Fleet and Mission 
    # Reference: https://www.eviation.com/wp-content/uploads/2022/09/Eviation-First-Flight-Press-Release-9.27.22.docx-1.pdf

In [72]:
# 1.1 Create unique aircrafts (Table 1: dim_aircraft)
    #Assumptions:
        # All aircraft are identical models with the same 850 kWh energy ceiling.
        # All aircraft have 100% SoH and start each mission at 100% SoC.
        # A static environment, temp is constant throughout the flight.
        # Introduce a A/B test variable.

aircraft_list = [f'N{i:03d}AL' for i in range(101, 111)]
dim_aircraft = pd.DataFrame({
    'tail_number': aircraft_list,
    'model': 'Alice_V1',
    'battery_capacity_kwh': 850,
    'max_payload_lbs': 2500,
    'firmware_version': np.random.choice(['v2.1', 'v2.2'], 10)
})

dim_aircraft.to_csv('aviation_data/dim_aircraft.csv', index=False)

print(dim_aircraft.shape)
dim_aircraft.head()

(10, 5)


,tail_number,model,battery_capacity_kwh,max_payload_lbs,firmware_version
0,N101AL,Alice_V1,850,2500,v2.2
1,N102AL,Alice_V1,850,2500,v2.2
2,N103AL,Alice_V1,850,2500,v2.1
3,N104AL,Alice_V1,850,2500,v2.2
4,N105AL,Alice_V1,850,2500,v2.1


In [73]:
# 1.2 Create 100 random flight missions -- （Table 2: flight_summary)
    # by assigning variable payloads and ambient temperatures
    # Assumption:
        #   payload_lbs refers to Useful Payload
        #   Gross Revenue per Sortie, calculated based on a regional express freight rate ($1.2 - $2.8/lb).
        #   Outside Air Temperature is assigned at the flight level and assumed constant.


num_flights = 100 

flight_summary = pd.DataFrame({
    'flight_id': [f'FL-{i:05d}' for i in range(1, num_flights + 1)],
    'tail_number': np.random.choice(dim_aircraft['tail_number'], num_flights),
    'payload_lbs': np.random.uniform(500, 2500, num_flights),
    'ambient_temp_c': np.random.uniform(-5, 45, num_flights),
    'ticket_revenue': np.random.uniform(3000, 7000, num_flights)
})


flight_summary.to_csv('aviation_data/flight_summary.csv', index=False)

print(flight_summary.shape)
flight_summary.head()

(100, 5)


,flight_id,tail_number,payload_lbs,ambient_temp_c,ticket_revenue
0,FL-00001,N109AL,963.911348,38.328565,5121.873710
1,FL-00002,N106AL,2478.888725,8.199876,5530.362245
2,FL-00003,N108AL,1516.209810,44.750618,4227.796568
3,FL-00004,N103AL,2293.328116,44.454490,3970.331914
4,FL-00005,N106AL,2026.038966,-2.234036,4184.185553


In [ ]:
# 1.3 Simulate 75-minute mission profiles (4,500s) to stress-test the 850kWh energy system -- Table 3: Fact_Telemetry_HighFreq 
    #Assumption：
        # Constant battery's internal resistance
        # Consolidate Phases: 
            # CLIMB: Includes Take-off and climb.  -- peak thermal and electrical stress period (1400kW scale).
            # CRUISE: Steady-state at 15,000 ft. -- longest duration
            # LANDING: Descent and landing, power-down phase. -- minimal used



telemetry_list = [] 
total_duration = 4500 

for index, flight in flight_summary.iterrows():
    f_id = flight['flight_id']
    payload = flight['payload_lbs']
    temp = flight['ambient_temp_c']
    payload_factor = (payload / 2500) * 0.005
    temp_factor = abs(temp - 25) * 0.0001
    current_soc = 100.0
    
    # Use (If/Else) that changes the aircraft's state based on the current time (by second)
    for second in range(total_duration):
        
        if second < 450:
            phase, altitude, phase_load = 'CLIMB', second * 33.3, 0.025
        elif second < 3800:
            phase, altitude, phase_load = 'CRUISE', 15000, 0.018
        else:
            phase, altitude, phase_load = 'LANDING', 15000 - ((second - 3800) * 22), 0.008

        drain = (phase_load + payload_factor + temp_factor)
        current_soc -= drain
        
        # append "for second"
        telemetry_list.append({
            'flight_id': f_id,
            'timestamp_sec': second,
            'altitude_ft': max(0, altitude),
            'battery_soc_pct': round(current_soc, 2),
            'phase': phase
        })

fact_telemetry_high_freq = pd.DataFrame(telemetry_list)
fact_telemetry_high_freq.to_csv('aviation_data/fact_telemetry_high_freq.csv', index=False)

print({len(fact_telemetry_high_freq)})
print(f" Safety Range: {fact_telemetry_high_freq['battery_soc_pct'].min()}% to {fact_telemetry_high_freq['battery_soc_pct'].max()}%")

fact_telemetry_high_freq.sample(10)

{450000}
 Safety Range: -8.35% to 99.97%


,flight_id,timestamp_sec,altitude_ft,battery_soc_pct,phase
250135,FL-00056,2635,15000.0,43.16,CRUISE
264444,FL-00059,3444,15000.0,16.68,CRUISE
296648,FL-00066,4148,7344.0,8.28,LANDING
333518,FL-00075,518,15000.0,85.55,CRUISE
208372,FL-00047,1372,15000.0,64.44,CRUISE
73599,FL-00017,1599,15000.0,64.36,CRUISE
415853,FL-00093,1853,15000.0,57.28,CRUISE
111379,FL-00025,3379,15000.0,24.04,CRUISE
223720,FL-00050,3220,15000.0,28.13,CRUISE
197230,FL-00044,3730,15000.0,15.76,CRUISE
